# RQ1 — Adaptive tokenization under the existing CNN.
#
# Notebook order (plan §§3–11):
#   1. Reproduce the frozen v1 environment (split, corruption, tokenizers).
#   2. Re-run the six fixed-tokenizer CNN comparison (same conditions as v1).
#   3. Phase 1 diagnostics: instability features vs. failure.
#   4. Phase 2 post-hoc routing: oracle target, threshold router, learned
#      routers, lambda sweep, robustness-compute frontier.
#   5. Phase 3: BPE-dropout baseline + OOD corruption-type holdout.
#   6. Failure analysis and artifact export.
#
# Run on the same machine/setup that produced the v1 results.

# # Typo Robustness v2 — RQ1 (CNN backbone)
#
# Research question: can observable tokenization instability predict when
# coarse tokenization fails, allowing a compact router to adapt representation
# granularity per input?

In [ ]:
import gc
import os

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import pointbiserialr

from common import (
    ARTIFACTS_DIR,
    BPE_DROPOUT_PROB,
    BPE_SEQUENCE_LENGTHS,
    CORRUPTION_LEVELS,
    CORRUPTIONS,
    FIGURES_DIR,
    LAMBDA_SWEEP,
    MODEL_ORDER,
    RESULTS_DIR,
    ROUTER_EXPERTS,
    TRAINING_SEEDS,
    BPEDropoutTrainingSequence,
    build_cnn,
    build_expert_probability_table,
    build_nested_test_sets,
    build_tokenization_suite,
    compute_instability_features,
    config_snapshot,
    enable_determinism,
    ensure_dirs,
    evaluate_model,
    expert_losses,
    fit_learned_router,
    fragmentation_threshold_router,
    label_failure_modes,
    load_test_set,
    load_train_val_set,
    make_bpe_dropout_tokenizer,
    make_train_val_split,
    make_word_vocab_index,
    mcnemar_holm_table,
    measure_cpu_latency_ms_per_sample,
    new_run_id,
    oracle_routing_target,
    route_probabilities,
    ROUTER_FEATURES,
    routing_regret,
    run_fixed_comparison,
    save_artifacts,
    save_figure,
    save_json,
    save_split_ids,
    summarize_routed_run,
    train_model,
)

enable_determinism()
ensure_dirs()
sns.set_theme(style="whitegrid")

RUN_ID = new_run_id()
print("RUN_ID:", RUN_ID)
save_json(
    config_snapshot(RUN_ID, backbone="cnn"),
    os.path.join(ARTIFACTS_DIR, f"config_v2_cnn_{RUN_ID}.json"),
)

# ## 1. Frozen data pipeline (identical to v1)

In [ ]:
train_val_set = load_train_val_set()
train_df, val_df = make_train_val_split(train_val_set)
test_df = load_test_set()

assert len(train_df) == 12000 and len(val_df) == 2000 and len(test_df) == 2000
save_split_ids(train_df, val_df, RUN_ID)
train_df["Class Index"].value_counts()

In [ ]:
suite = build_tokenization_suite(train_df["Description"].to_numpy())
word_vocab = make_word_vocab_index(suite)
save_artifacts(suite, RUN_ID)

# ## 2. Nested corruption test sets

In [ ]:
test_sets, corruption_summary_df = build_nested_test_sets(
    source_df=test_df,
    corruption_levels=CORRUPTION_LEVELS,
    corruption_seed=0,
)
display(corruption_summary_df)

# ## 3. Fixed-tokenizer CNN comparison (six models x nine seeds)

In [ ]:
results = run_fixed_comparison(
    backbone="cnn",
    suite=suite,
    train_df=train_df,
    val_df=val_df,
    test_sets=test_sets,
    run_id=RUN_ID,
)
metrics_df = results["metrics"]
resource_usage_df = results["resource_usage"]
predictions_long = results["predictions"]

display(
    metrics_df.groupby(["model", "corruption_level"], as_index=False).agg(
        f1_mean=("f1_macro", "mean"), f1_std=("f1_macro", "std")
    )
)

# ### 3.1 Per-expert CPU latency (single-sample — plan §14)

In [ ]:
calibration_texts = test_df["Description"].iloc[:64].to_numpy()
latency_ms = {}
for model_name in MODEL_ORDER:
    for seed in TRAINING_SEEDS:
        model_path = os.path.join(ARTIFACTS_DIR, "models", f"cnn_{model_name}_seed{seed}_{RUN_ID}.keras")
        model = keras.models.load_model(model_path)
        latency_ms[(model_name, seed)] = measure_cpu_latency_ms_per_sample(
            model, suite, model_name, calibration_texts
        )
        del model
        keras.backend.clear_session()

latency_df = pd.DataFrame(
    [(m, s, v) for (m, s), v in latency_ms.items()],
    columns=["model", "seed", "latency_ms_per_sample"],
)
latency_df.to_csv(os.path.join(RESULTS_DIR, f"cnn_latency_{RUN_ID}.csv"), index=False)
display(latency_df.groupby("model")["latency_ms_per_sample"].describe())

# Mean per-sample latency per expert feeds the oracle cost term C_m.
cost_per_sample_ms = (
    latency_df.groupby("model")["latency_ms_per_sample"].mean().loc[ROUTER_EXPERTS].to_dict()
)
print("Expert costs C_m (ms/sample):", cost_per_sample_ms)

# ## 4. Phase 1 — Instability diagnostics (plan §5)

In [ ]:
# Features for every corrupted test input at every corruption level.
feature_frames = []
for corruption_level, test_set in test_sets.items():
    feats = compute_instability_features(test_set["Description"].to_numpy(), suite, word_vocab)
    feats.insert(0, "corruption_level", corruption_level)
    feats.insert(0, "sample_id", test_set.index.to_numpy())
    feature_frames.append(feats)
features_all = pd.concat(feature_frames, ignore_index=True)
features_all.to_csv(os.path.join(RESULTS_DIR, f"instability_features_{RUN_ID}.csv"), index=False)

# ### 4.1 Does corruption increase measured instability? (H1 mechanism check)

In [ ]:
instability_by_level = features_all.groupby("corruption_level")[
    [
        "bpe_tokens_per_word",
        "fraction_words_split_2plus",
        "max_pieces_per_word",
        "word_oov_fraction",
        "sequence_length",
    ]
].mean()
display(instability_by_level)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col, title in [
    (axes[0], "bpe_tokens_per_word", "BPE pieces per word"),
    (axes[1], "fraction_words_split_2plus", "Fraction of words split into 2+ pieces"),
    (axes[2], "word_oov_fraction", "Word-level OOV fraction"),
]:
    sns.lineplot(data=features_all, x="corruption_level", y=col, ax=ax)
    ax.set_title(title)
save_figure(fig, "instability_vs_corruption_level", RUN_ID)
plt.show()

# ### 4.2 Do instability features predict per-example failure?

In [ ]:
diagnostics_df = features_all.copy()
for expert in ROUTER_EXPERTS:
    sub = predictions_long[predictions_long["model"].eq(expert)]
    correct_map = sub.set_index(["sample_id", "corruption_level"])["correct"]
    keys = list(diagnostics_df[["sample_id", "corruption_level"]].itertuples(index=False, name=None))
    diagnostics_df[f"correct_{expert}"] = [correct_map.get(k) for k in keys]

diagnostics_df.to_csv(os.path.join(RESULTS_DIR, f"failure_diagnostics_{RUN_ID}.csv"), index=False)

correlation_rows = []
for feature in ROUTER_FEATURES:
    for expert in ROUTER_EXPERTS:
        r, p = pointbiserialr(diagnostics_df[f"correct_{expert}"], diagnostics_df[feature])
        correlation_rows.append({"feature": feature, "expert": expert, "pointbiserial_r": r, "p_value": p})
correlations_df = pd.DataFrame(correlation_rows).sort_values("pointbiserial_r")
correlations_df.to_csv(os.path.join(RESULTS_DIR, f"feature_failure_correlations_{RUN_ID}.csv"), index=False)
display(correlations_df.head(15))

# ### 4.3 Can features predict when character-level processing pays off?

In [ ]:
prob_table_full = build_expert_probability_table(predictions_long, experts=list(MODEL_ORDER))
loss_matrix = expert_losses(prob_table_full, experts=list(MODEL_ORDER))
expert_list_full = list(MODEL_ORDER)
gap_word_char = loss_matrix[:, expert_list_full.index("word")] - loss_matrix[:, expert_list_full.index("char")]

gap_df = prob_table_full[["sample_id", "seed", "corruption_level"]].copy()
gap_df["loss_gap_word_minus_char"] = gap_word_char
gap_df.to_csv(os.path.join(RESULTS_DIR, f"word_char_loss_gap_{RUN_ID}.csv"), index=False)

gap_merged = gap_df.merge(features_all, on=["sample_id", "corruption_level"], how="left")
gap_correlations = {
    feature: float(np.corrcoef(gap_merged[feature].fillna(0), gap_merged["loss_gap_word_minus_char"])[0, 1])
    for feature in ROUTER_FEATURES
}
display(pd.Series(gap_correlations, name="corr_with_loss_gap_word_minus_char").sort_values())

# **Decision gate (plan §5):** if no cheap feature correlates meaningfully
# with failure or with the word-char loss gap, stop rather than force an
# adaptive model. The tables above are that evidence.

# ## 5. Phase 2 — Post-hoc routers over word / BPE-500 / char (plan §§6–8)

In [ ]:
prob_table = build_expert_probability_table(predictions_long, experts=ROUTER_EXPERTS)

oracle_target_df = oracle_routing_target(prob_table, cost_per_sample_ms, LAMBDA_SWEEP, ROUTER_EXPERTS)
oracle_target_df.to_csv(os.path.join(RESULTS_DIR, f"oracle_routing_target_{RUN_ID}.csv"), index=False)

# Feature matrix aligned 1:1 with prob_table row order (left merge keeps order).
features_for_routing = prob_table[["sample_id", "seed", "corruption_level"]].merge(
    features_all, on=["sample_id", "corruption_level"], how="left"
)[ROUTER_FEATURES]
assert len(features_for_routing) == len(prob_table)

# Mid-sweep lambda defines the router's training target (plan §7 sweep).
MID_LAMBDA = LAMBDA_SWEEP[len(LAMBDA_SWEEP) // 2]
oracle_labels = oracle_target_df[oracle_target_df["lambda"].eq(MID_LAMBDA)]["oracle_expert"].to_numpy()
assert len(oracle_labels) == len(prob_table)

# Train routers on a subset of seeds; evaluate on the held-out seeds.
ROUTER_TRAIN_SEEDS = TRAINING_SEEDS[:6]  # prespecified split of seeds
router_train_mask = prob_table["seed"].isin(ROUTER_TRAIN_SEEDS).to_numpy()

learned_routers = {}
for router_type in ("logistic_regression", "decision_tree", "gradient_boosting"):
    learned_routers[router_type] = fit_learned_router(
        features_for_routing.iloc[router_train_mask],
        oracle_labels[router_train_mask],
        router_type=router_type,
        random_state=0,
    )

In [ ]:
frontier_rows = []


def record_frontier_point(system_name, routed_df, lam=np.nan):
    summary = summarize_routed_run(routed_df)
    summary.update({"system": system_name, "lambda": lam})
    frontier_rows.append(summary)


# Fixed single-expert systems (all examples).
for expert in ROUTER_EXPERTS:
    record_frontier_point(f"fixed_{expert}", route_probabilities(prob_table, [expert] * len(prob_table)))

# Oracle router across the lambda sweep.
for lam in LAMBDA_SWEEP:
    oracle_at_lam = oracle_target_df[oracle_target_df["lambda"].eq(lam)]
    record_frontier_point("oracle", route_probabilities(prob_table, oracle_at_lam["oracle_expert"].to_numpy()), lam)

# Simple fragmentation-threshold router: sweep thresholds to trace its curve.
threshold_grid = np.arange(0.02, 0.61, 0.04)
for t in threshold_grid:
    thr_routing = fragmentation_threshold_router(features_for_routing, thresholds=(t,))
    record_frontier_point("fragmentation_threshold", route_probabilities(prob_table, np.asarray(thr_routing)))

# Learned routers: evaluated on the held-out seeds.
held_out_mask = ~router_train_mask
for router_type, estimator in learned_routers.items():
    routing = estimator.predict(features_for_routing.iloc[held_out_mask].to_numpy())
    record_frontier_point(f"learned_{router_type}", route_probabilities(prob_table.iloc[held_out_mask], routing))

frontier_df = pd.DataFrame(frontier_rows)
frontier_df.to_csv(os.path.join(RESULTS_DIR, f"routing_frontier_{RUN_ID}.csv"), index=False)
display(frontier_df)

# **Note:** fixed experts and the oracle are evaluated on ALL seeds; learned
# routers are evaluated on their held-out seeds only (they were trained on the
# other six). Frontier points across these groups are indicative; the paired
# statistical comparison below is the controlled one.

# ### 5.1 Key figure: Macro-F1 vs compute (plan §8)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
palette = sns.color_palette("tab10")
for i, system in enumerate(frontier_df["system"].unique()):
    sub = frontier_df[frontier_df["system"].eq(system)]
    is_curve = system in ("oracle", "fragmentation_threshold")
    marker = "o" if is_curve else "D"
    linestyle = "-" if is_curve else "none"
    ax.plot(
        sub["mean_active_sequence_length"],
        sub["macro_f1"],
        marker=marker,
        linestyle=linestyle,
        color=palette[i % len(palette)],
        label=system,
        markersize=4,
    )
ax.set_xlabel("Mean active sequence length (tokens)")
ax.set_ylabel("Macro F1 (all corruption levels pooled)")
ax.set_title("Robustness-compute frontier — CNN backbone")
ax.legend(fontsize=8)
save_figure(fig, "cnn_frontier_f1_vs_sequence_cost", RUN_ID)
plt.show()

# ### 5.2 Routing regret vs oracle (plan §8)

In [ ]:
logistic_router = learned_routers["logistic_regression"]
learned_routing_full = logistic_router.predict(features_for_routing.to_numpy())
learned_routed = route_probabilities(prob_table, learned_routing_full)

regret_df = routing_regret(learned_routed, oracle_target_df, MID_LAMBDA)
regret_summary = {
    "lambda": MID_LAMBDA,
    "mean_regret": float(regret_df["regret"].mean()),
    "median_regret": float(regret_df["regret"].median()),
    "fraction_optimal_choice": float(regret_df["is_optimal_choice"].mean()),
    "p99_regret": float(regret_df["regret"].quantile(0.99)),
}
display(pd.Series(regret_summary))
regret_df.to_csv(os.path.join(RESULTS_DIR, f"cnn_routing_regret_{RUN_ID}.csv"), index=False)

# ### 5.3 Paired statistics: adaptive vs best fixed (plan §13)

In [ ]:
systems_predictions = [
    predictions_long[predictions_long["model"].isin(ROUTER_EXPERTS)].rename(columns={"model": "system"}),
]
adaptive_pred = learned_routed[["sample_id", "seed", "corruption_level", "true_class", "predicted_class", "correct"]].copy()
adaptive_pred.insert(1, "system", "learned_logistic_regression")
systems_predictions.append(adaptive_pred)

all_systems_df = pd.concat(systems_predictions, ignore_index=True)
system_order = [*ROUTER_EXPERTS, "learned_logistic_regression"]
mcnemar_systems_df = mcnemar_holm_table(
    all_systems_df,
    system_order=system_order,
    seeds=[s for s in TRAINING_SEEDS if s not in ROUTER_TRAIN_SEEDS],  # fair: unseen by router
    corruption_levels=CORRUPTION_LEVELS,
    csv_name=f"mcnemar_systems_cnn_{RUN_ID}.csv",
)
display(mcnemar_systems_df[mcnemar_systems_df["significant_holm_0.05"]])

# ## 6. Phase 3a — BPE dropout baseline (plan §10)

In [ ]:
dropout_tokenizer = make_bpe_dropout_tokenizer(suite.bpe_tokenizers[500], BPE_DROPOUT_PROB)

from tensorflow.keras.utils import Sequence as KerasSequence

KerasSequence.register(BPEDropoutTrainingSequence)

train_Y = (train_df["Class Index"] - 1).to_numpy()
val_Y = (val_df["Class Index"] - 1).to_numpy()
val_X_bpe500 = suite.vectorize("bpe_500", val_df["Description"].to_numpy())

bpe_dropout_metrics: list[dict] = []
bpe_dropout_predictions: list[pd.DataFrame] = []

for seed in TRAINING_SEEDS:
    print(f"\n[BPE-dropout p={BPE_DROPOUT_PROB}] Training bpe_500_dropout, seed={seed}")
    model, history, _stats = train_model(
        model_builder=lambda: build_cnn(BPE_SEQUENCE_LENGTHS[500], suite.bpe_tokenizers[500].get_vocab_size(), 32),
        train_X=BPEDropoutTrainingSequence(train_df["Description"].to_numpy(), train_Y, dropout_tokenizer),
        train_Y=None,
        val_X=(val_X_bpe500, val_Y),
        val_Y=None,
        seed=seed,
        batch_size=1,
    )
    metric_rows, prediction_frames = evaluate_model(
        model=model,
        model_name="bpe_500",
        seed=seed,
        suite=suite,
        test_sets=test_sets,
        system_label="bpe_500_dropout",
    )
    bpe_dropout_metrics.extend(metric_rows)
    bpe_dropout_predictions.extend(prediction_frames)
    model.save(os.path.join(ARTIFACTS_DIR, "models", f"cnn_bpe_500_dropout_seed{seed}_{RUN_ID}.keras"))
    del model
    keras.backend.clear_session()
    gc.collect()

pd.DataFrame(bpe_dropout_metrics).to_csv(
    os.path.join(RESULTS_DIR, f"cnn_bpe_dropout_metrics_{RUN_ID}.csv"), index=False
)
pd.concat(bpe_dropout_predictions, ignore_index=True).to_csv(
    os.path.join(RESULTS_DIR, f"cnn_bpe_dropout_predictions_{RUN_ID}.csv"), index=False
)

# ### 6.1 Fixed BPE vs BPE dropout vs experts

In [ ]:
comparison_metrics = pd.concat(
    [
        metrics_df[metrics_df["model"].isin([*ROUTER_EXPERTS, "bpe_1000"])],
        pd.DataFrame(bpe_dropout_metrics),
    ],
    ignore_index=True,
)
degradation_comparison = comparison_metrics.groupby(["model", "corruption_level"], as_index=False)["f1_macro"].mean()

fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(data=degradation_comparison, x="corruption_level", y="f1_macro", hue="model", marker="o", ax=ax)
ax.set_title("Fixed tokenizers vs BPE dropout — Macro F1 by corruption level")
save_figure(fig, "cnn_bpe_dropout_comparison", RUN_ID)
plt.show()

# ## 7. Phase 3b — OOD corruption-type holdout (plan §11 Option A)

In [ ]:
HOLDOUT_FAMILY = "substitution"  # prespecified before looking at any OOD result
print(f"Router trained on mixed corruptions; OOD sets corrupt ONLY with '{HOLDOUT_FAMILY}'.")

ood_test_sets, ood_summary = build_nested_test_sets(
    source_df=test_df,
    corruption_levels=CORRUPTION_LEVELS,
    corruption_seed=0,
    allowed_corruptions=[HOLDOUT_FAMILY],
)
ood_summary.to_csv(os.path.join(RESULTS_DIR, f"ood_corruption_summary_{RUN_ID}.csv"), index=False)

ood_feature_frames = []
for corruption_level, ood_set in ood_test_sets.items():
    feats = compute_instability_features(ood_set["Description"].to_numpy(), suite, word_vocab)
    feats.insert(0, "corruption_level", corruption_level)
    feats.insert(0, "sample_id", ood_set.index.to_numpy())
    ood_feature_frames.append(feats)
ood_features_all = pd.concat(ood_feature_frames, ignore_index=True)

In [ ]:
# Expert predictions on the OOD sets from the already-trained models.
ood_prediction_frames: list[pd.DataFrame] = []
for model_name in ROUTER_EXPERTS:
    for seed in TRAINING_SEEDS:
        model_path = os.path.join(ARTIFACTS_DIR, "models", f"cnn_{model_name}_seed{seed}_{RUN_ID}.keras")
        model = keras.models.load_model(model_path)
        for corruption_level, ood_set in ood_test_sets.items():
            X = suite.vectorize(model_name, ood_set["Description"].to_numpy())
            probs = model.predict(X, batch_size=64, verbose=0)
            preds = probs.argmax(axis=1)
            y_true = (ood_set["Class Index"].to_numpy() - 1).astype(int)
            df = pd.DataFrame(
                {
                    "sample_id": ood_set.index.to_numpy(),
                    "model": model_name,
                    "seed": seed,
                    "corruption_level": corruption_level,
                    "true_class": y_true + 1,
                    "predicted_class": preds + 1,
                    "correct": (preds == y_true).astype(int),
                }
            )
            for class_id in range(4):
                df[f"prob_class_{class_id + 1}"] = probs[:, class_id]
            ood_prediction_frames.append(df)
        del model
        keras.backend.clear_session()
        gc.collect()

ood_predictions_long = pd.concat(ood_prediction_frames, ignore_index=True)
ood_predictions_long.to_csv(os.path.join(RESULTS_DIR, f"cnn_ood_expert_predictions_{RUN_ID}.csv"), index=False)

In [ ]:
# In-distribution-trained router applied unchanged — that IS the OOD test.
ood_prob_table = build_expert_probability_table(ood_predictions_long, experts=ROUTER_EXPERTS)
ood_features_for_routing = ood_prob_table[["sample_id", "seed", "corruption_level"]].merge(
    ood_features_all, on=["sample_id", "corruption_level"], how="left"
)[ROUTER_FEATURES]

ood_frontier_rows = []
for expert in ROUTER_EXPERTS:
    summary = summarize_routed_run(route_probabilities(ood_prob_table, [expert] * len(ood_prob_table)))
    ood_frontier_rows.append({"system": f"fixed_{expert}", **summary})

ood_learned_routing = logistic_router.predict(ood_features_for_routing.to_numpy())
summary = summarize_routed_run(route_probabilities(ood_prob_table, ood_learned_routing))
ood_frontier_rows.append({"system": "learned_logistic_regression", **summary})

ood_frontier_df = pd.DataFrame(ood_frontier_rows)
ood_frontier_df.to_csv(os.path.join(RESULTS_DIR, f"cnn_routing_frontier_ood_{RUN_ID}.csv"), index=False)
display(ood_frontier_df)

# ## 8. Targeted failure analysis (plan §15)

In [ ]:
correct_by_expert = {
    expert: predictions_long[predictions_long["model"].eq(expert)]
    .set_index(["sample_id", "seed", "corruption_level"])["correct"]
    for expert in ("word", "char")
}
failure_input = regret_df.copy()
keys = list(failure_input[["sample_id", "seed", "corruption_level"]].itertuples(index=False, name=None))
failure_input["correct_word"] = [correct_by_expert["word"].get(k) for k in keys]
failure_input["correct_char"] = [correct_by_expert["char"].get(k) for k in keys]
failure_input = failure_input.merge(
    features_all, on=["sample_id", "corruption_level"], how="left"
)

labeled_failures = label_failure_modes(failure_input)
failure_counts = labeled_failures[labeled_failures["failure_mode"].ne("")]["failure_mode"].value_counts()
display(failure_counts)
failure_counts.to_csv(os.path.join(RESULTS_DIR, f"cnn_failure_mode_counts_{RUN_ID}.csv"))

# ## 9. Persist run manifest
manifest = {
    "run_id": RUN_ID,
    "backbone": "cnn",
    "router_train_seeds": ROUTER_TRAIN_SEEDS,
    "mid_lambda": MID_LAMBDA,
    "holdout_family": HOLDOUT_FAMILY,
    "regret_summary": regret_summary,
}
save_json(manifest, os.path.join(RESULTS_DIR, f"run_manifest_cnn_{RUN_ID}.json"))
print("RQ1 notebook complete.")